## Ingestão do tracking mobile: Mongo para raw
Copia a coleção `tracking_events` do MongoDB para `workspace.raw.tracking_events`, com todas as colunas como texto (cópia fiel, sem inferir tipo). A conversão de tipos é feita na `transformations/apollo_cleaned_tracking.py`, executada pela pipeline `Apollo_BI`. A cada execução a tabela é refeita por completo.

In [0]:
%pip install pymongo[srv] python-dotenv

In [0]:
dbutils.library.restartPython()

#### Configuração
Este notebook não depende do core do Neon: só precisa de `MONGO_URI` (e `MONGO_DB`, se o nome do banco não estiver na URI) no `.env`.

In [0]:
import os
import re
from datetime import date, datetime
from dotenv import load_dotenv, find_dotenv
from pymongo import MongoClient
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

load_dotenv(find_dotenv(usecwd=True))

CATALOG = "workspace"
COLLECTION = "tracking_events"

client = MongoClient(os.getenv("MONGO_URI"), serverSelectionTimeoutMS=20000)
db_name = os.getenv("MONGO_DB")
mongo_db = client[db_name] if db_name else client.get_default_database()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.raw")

#### Ler a coleção

##### Achatar um documento

In [0]:
def flatten(doc, prefix=""):
    out = {}
    for key, value in doc.items():
        name = "mongo_id" if (prefix == "" and key == "_id") else f"{prefix}{key}"
        name = re.sub(r"[ ,;{}()\n\t=.]", "_", name)
        if isinstance(value, dict):
            out.update(flatten(value, prefix=f"{name}_"))
        elif value is None:
            out[name] = None
        elif isinstance(value, (datetime, date)):
            out[name] = value.isoformat()
        else:
            out[name] = str(value)
    return out

##### Ler a coleção para um DataFrame

In [0]:
def read_mongo(collection):
    docs = [flatten(d) for d in mongo_db[collection].find({})]
    if not docs:
        raise ValueError(f"A coleção '{collection}' não existe ou está vazia")
    columns = list(dict.fromkeys(c for d in docs for c in d))
    schema = StructType([StructField(c, StringType(), True) for c in columns])
    return spark.createDataFrame([tuple(d.get(c) for c in columns) for d in docs], schema)

#### Salvar na raw

In [0]:
df = read_mongo(COLLECTION).withColumn("ingested_at", F.current_timestamp())
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.raw.{COLLECTION}")
print(COLLECTION, "ok:", df.count(), "documentos")

#### Conferir

In [0]:
display(spark.table(f"{CATALOG}.raw.{COLLECTION}").limit(20))